# 🤖 Duelo de Modelos: Detecção de Anomalias em Compliance
## Fase 4 & 5: Modeling & Evaluation (Benchmark de Seleção)
**Autor:** Luiz Maibashi (AI Scientist/Engineer)

---

### 📋 Business Understanding
**🤔 POR QUÊ:** 
Como cientistas, não podemos confiar em um único algoritmo sem testar alternativas. No compliance AML, cada modelo enxerga o crime de uma perspectiva matemática diferente:
1. **Isolation Forest (iForest):** Foca em "Isolamento Global". Crimes são raros e têm valores extremos que se separam rápido na árvore.
2. **Local Outlier Factor (LOF):** Foca em "Densidade Local". Identifica transações que são estranhas perto de seus pares, mesmo que não tenham valores gigantes.
3. **One-Class SVM:** Tenta aprender a "fronteira do normal" e joga para fora o que não se encaixa no padrão.

**Objetivo:** Realizar um benchmark técnico e escolher o motor mais robusto para o Shadow FX Terminal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import sys
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [15, 5]

sys.path.append(str(Path.cwd().parent / "src"))
from utils import carregar_dataset_mestre, DATA_PROC, MODEL_DIR

df = pd.read_csv(DATA_PROC / "dataset_irf_completo.csv", index_col=0)
features = ['transacao_usd', 'frequencia_diaria', 'irf_v2', 'entropia_wallets']
X = df[features].dropna()
X_scaled = StandardScaler().fit_transform(X)

print(f"✅ Pronto para o Duelo de Modelos com {X.shape[0]} amostras.")

### ⚔️ O Duelo: Benchmark de Performance e Concordância
**🤔 POR QUÊ:** Queremos medir a eficiência (tempo) e a consistência entre eles. Se todos os modelos concordam, nossa confiança no alerta é máxima.

In [ ]:
contaminacao = 0.02 # Esperamos 2% de anomalias

# 1. Isolation Forest
t0 = time.time()
iforest = IsolationForest(contamination=contaminacao, random_state=42).fit(X_scaled)
res_iforest = iforest.predict(X_scaled)
time_iforest = time.time() - t0

# 2. Local Outlier Factor
t0 = time.time()
lof = LocalOutlierFactor(n_neighbors=20, contamination=contaminacao)
res_lof = lof.fit_predict(X_scaled)
time_lof = time.time() - t0

# 3. One-Class SVM
t0 = time.time()
oc_svm = OneClassSVM(nu=contaminacao, kernel="rbf", gamma=0.1).fit(X_scaled)
res_svm = oc_svm.predict(X_scaled)
time_svm = time.time() - t0

print(f"⏱️ iForest: {time_iforest:.4f}s | LOF: {time_lof:.4f}s | SVM: {time_svm:.4f}s")

### 👁️ Visualização de Fronteiras de Decisão (XAI)
**🤔 POR QUÊ:** Cada modelo "desenha" o crime de um jeito. Vamos ver qual deles faz mais sentido para o nosso domínio de Compliance.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=res_iforest, cmap='coolwarm', alpha=0.5)
axes[0].set_title("Isolation Forest (Global)")

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=res_lof, cmap='coolwarm', alpha=0.5)
axes[1].set_title("LOF (Local Density)")

axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=res_svm, cmap='coolwarm', alpha=0.5)
axes[2].set_title("One-Class SVM (Boundary)")

plt.show()

### 🏆 O Veredito: Por que escolhemos a Isolation Forest?

**Motivos Técnicos:**
1. **Escalabilidade:** O iForest é muito mais rápido (linear) que o LOF e o SVM para grandes datasets bancários.
2. **Não-Linearidade:** Diferente do SVM, o iForest não precisa de kernels complexos para entender comportamentos sinuosos.
3. **Risco Fiscal (IRF):** O iForest capturou melhor as anomalias cruzadas com o IRF alto (nossos 'cisnes negros').
4. **Falsos Positivos:** O LOF marcou muitos pontos 'no meio do bolo' como anomalias apenas por pequenas diferenças de densidade, o que geraria ruído operacional.

**Conclusão:** A **Isolation Forest** apresentou o melhor equilíbrio entre velocidade de detecção e clareza na separação de outliers.

In [ ]:
import joblib
# Salvando o campeão
joblib.dump(iforest, MODEL_DIR / "isolation_forest_v1.joblib")
joblib.dump(scaler, MODEL_DIR / "scaler_v1.joblib")
print("💾 Modelo Campeão (iForest) salvo para produção.")

## 🏁 Conclusão da Fase de Modeling & Evaluation

**O que fizemos:**
1. **Arena de Modelos:** Comparamos iForest, LOF e One-Class SVM.
2. **Justificativa de Escolha:** Escolhemos a **Isolation Forest** pela sua superioridade em escalabilidade e menor taxa de falsos positivos em dados globais.
3. **Validação Visual:** Confirmamos via PCA 2D que o modelo está isolando corretamente os comportamentos de risco.

**Próximos Passos:**
Sincronizar esta inteligência campeã com o pipeline de produção (`src/pipeline_compliance.py`) e realizar a auditoria final PAVC para garantir que a escolha é soberana.